# Stage 2 → Stage 3 walkthrough: one filing, raw HTML to search results

Every step here calls the project's own functions — nothing is reimplemented — so
what you see is exactly what `fc embed` and `fc build-index` do, one filing at a time.

**Prerequisites**
- `uv sync --group notebook` (ipykernel + pandas), then pick the `.venv` kernel
- Filings downloaded (`fc fetch-filings`), so the EDGAR cache is warm
- Ollama running with `nomic-embed-text` (cells 7–9, 12)
- `fc embed` and `fc build-index` already run, and OpenSearch up (cells 8–12)

Companion reading: `docs/stage2_filing_text.md`, `docs/stage3_embeddings_index.md`.

## 0 · Setup — find the filing in the cache

`download_company` is the same call `fc fetch-filings` makes. With a warm cache it
reads from disk and makes **zero** requests to SEC — asserted below, not assumed.
This gives the real `FilingRef` (accession, report date, path) instead of one typed by hand.

`Settings` is never printed: it holds your EDGAR User-Agent contact, and this repo is public.

In [ ]:
import math
import os
from pathlib import Path

# .env, config/ and data/ are relative to the repo root, exactly as when running `fc`.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from filing_copilot.config import get_settings
from filing_copilot.edgar import EdgarClient
from filing_copilot.filings import download_company
from filing_copilot.structured import Corpus

TICKER = "SYF"          # try WFC or USB (10-K + Annual Report exhibit), JPM (referrals), C (page table)
FISCAL_YEAR = 2025

settings = get_settings()
corpus = Corpus.load(settings.corpus_path)
company = corpus.by_ticker(TICKER)

with EdgarClient(settings) as client:
    downloaded = download_company(client, company, years=3)
    assert client.request_count == 0, "cache was cold -- run `fc fetch-filings` first"

document = next(d for d in downloaded.documents if d.ref.fiscal_year == FISCAL_YEAR)
document.ref, document.path

## 1 · Raw HTML — what SEC actually serves

A 10-K is not a document with some markup around it. It is inline XBRL: every
figure wrapped in a tag, every paragraph in nested spans with inline styles.

In [ ]:
raw = document.path.read_bytes()
print(f"{len(raw):,} bytes")
print(f"style= attributes: {raw.count(b'style='):,}   <span: {raw.count(b'<span'):,}   <table: {raw.count(b'<table'):,}")
print()
print(raw[:1500].decode("utf-8", errors="replace"))

## 2 · Normalize — HTML to one clean string

`normalize()` is pure: bytes in, string out. Data tables become placeholders
(figures come from XBRL via SQL, never from text); layout tables keep their text;
block elements become newlines.

**Every `char_start`/`char_end` in the rest of the system indexes into this exact string.**

In [ ]:
from filing_copilot.filings import normalize

text = normalize(raw)
print(f"{len(raw):,} bytes of HTML  ->  {len(text):,} chars of text   ({len(raw) / len(text):.1f}x smaller)")
print(f"data tables replaced by placeholders: {text.count('[TABLE:'):,}")
print()
start = text.find("[TABLE:")
print(text[max(0, start - 600) : start + 200])

## 3 · Section — where each Item starts and ends

`build_filing_text` is what `fc sections`, `fc show` and `fc embed` all call. It
normalizes every document of the filing — the 10-K, plus an Annual Report exhibit
(EX-13) when the 10-K incorporates one — into **one** string, then locates items:

1. **item headings** in the 10-K;
2. **referrals** — a heading that only says "can be found in the Annual Report on
   pages 22 to 59" is followed to the content (`strategy = referral`);
3. **the filer's page table**, when no heading is found at all (`crossref_pages`);
4. **declared beats inferred** — a span the filer declared is carved out of any
   heading span it overlaps.

An item can have several spans. For SYF, only the page table works (its body
has no item headings).

In [ ]:
from filing_copilot.filings import build_filing_text

doc = build_filing_text(document, company.name)
text, sections = doc.text, doc.sections
print("documents:", [(p.name, p.start, p.end) for p in doc.parts])
pd.DataFrame(
    [
        {"item": s.item, "strategy": s.strategy, "spans": len(s.spans),
         "primary_start": s.char_start, "primary_length": s.length,
         "total_length": sum(end - start for start, end in s.spans)}
        for s in sections.values()
    ]
).sort_values("primary_start")

## 3b · When two items claim the same text

Filers sometimes declare the same pages for two items — Citigroup lists pages
64–120 under both Item 7 and Item 7A; SYF's Item 3 sits inside its Item 8.
`labelled_segments` cuts the filing at every span boundary and labels each piece
with **every** item that claims it. Each character is chunked once; a search
filtered to either item finds it.

In [ ]:
from filing_copilot.filings import labelled_segments

segments = labelled_segments(sections)
shared = [s for s in segments if len(s.items) > 1]
print(f"{len(segments)} segments, {len(shared)} claimed by more than one item")
pd.DataFrame([{"items": s.items, "start": s.start, "length": s.end - s.start} for s in shared])

## 4 · One section, read with your own eyes

"Located" only means a span of plausible length was found. Whether it holds the
right content is something only reading it can answer — this is the check `fc show` automates.

In [ ]:
risk = sections["1A"]
body = text[risk.char_start : risk.char_end]
print(f"Item 1A: chars {risk.char_start:,}-{risk.char_end:,} ({risk.length:,} long)\n")
print(body[:800])
print("\n[...]\n")
print(body[-500:])

## 5 · Chunk — ~800-token windows that never cross an item boundary

`item_aware` windows *within* each section; `fixed_window` ignores structure and is
the baseline Stage 4 measures against (ADR-0002). The assert is the property every
citation rests on: a chunk's offsets select exactly its text.

In [ ]:
from filing_copilot.filings import fixed_window, item_aware

chunks = item_aware(doc)

for chunk in chunks:
    assert text[chunk.char_start : chunk.char_end] == chunk.text   # the offset round trip

tokens = pd.Series([c.tokens for c in chunks])
print(f"item_aware: {len(chunks)} chunks   fixed_window: {len(fixed_window(doc))} chunks")
print(f"tokens per chunk: median {tokens.median():.0f}, p95 {tokens.quantile(0.95):.0f}")
print(pd.Series([c.items for c in chunks]).value_counts().sort_index().to_dict())

chunk = next(c for c in chunks if "1A" in c.items)
chunk

## ── Stage 2 ends here. Everything below is Stage 3. ──

## 6 · The embedding input — the exact string the model sees

Three layers, assembled in exactly one place:

```
"search_document: " + contextual_prefix + "\n\n" + chunk.text
```

The task prefix is outermost. nomic needs it, and **Ollama does not add it for you** —
omitting it degrades retrieval with no error at all.

In [ ]:
from dataclasses import replace

from filing_copilot.embed import NOMIC_EMBED_TEXT, SEARCH_DOCUMENT, document_input, prepare
from filing_copilot.filings import contextual_prefix

# The same model `fc embed` builds: nomic, at the configured width (768 by default).
model = replace(NOMIC_EMBED_TEXT, dimensions=settings.embedding_dimensions)
print(model)
print("contextual prefix:", contextual_prefix(doc, chunk.items))
print()
body = document_input(doc, chunk)       # prefix + separator + text  (what the pipeline hashes)
sent = prepare(model, SEARCH_DOCUMENT, body)
print(sent[:400])

## 7 · Encode — 768 numbers, length 1

Ollama returns vectors with a norm around 20. The encoder normalizes every vector to
unit length, because the index scores by inner product — which only equals cosine for unit vectors.

The **digest** is a SHA-256 of `sent` above. It is the vector's key in the cache.

In [ ]:
from filing_copilot.embed import OllamaEncoder, digest

encoder = OllamaEncoder(host=settings.ollama_host, model=model)
live = encoder.encode([body], task=SEARCH_DOCUMENT)[0]
key = digest(model, SEARCH_DOCUMENT, body)

print(f"{len(live)} dimensions, norm {math.sqrt(sum(v * v for v in live)):.6f}")
print("first 8:", [round(v, 4) for v in live[:8]])
print("digest:", key)

## 8 · The cache holds the same vector

`fc embed` stored this chunk's vector under the same digest. Cosine ≈ 1.0 shows the
cache is exactly what the model says — and why a changed input can never be served a stale vector:
a different input is a different digest.

In [ ]:
from filing_copilot.embed import EmbeddingCache

cache = EmbeddingCache(settings.embeddings_dir)
cached = cache.load(model, [key])[key]
print("cosine(live, cached) =", round(sum(a * b for a, b in zip(live, cached)), 6))
cache.stats(model)

## 9 · Semantic search by hand — what k-NN does before OpenSearch

Embed a *question* (note: `search_query`, not `search_document`), then take the dot
product with every one of this company's cached vectors and sort. That is all
nearest-neighbour search is. OpenSearch's HNSW index does the same thing approximately,
without comparing against every vector.

In [ ]:
from filing_copilot.embed import SEARCH_QUERY
from filing_copilot.index import manifest_path, read_manifest

QUESTION = "credit card net charge-off risk"
query_vector = encoder.encode([QUESTION], task=SEARCH_QUERY)[0]

rows = read_manifest(manifest_path(settings.manifests_dir, "item_aware", model.cache_key))
company_rows = [r for r in rows if r.ticker == TICKER]
vectors = cache.load(model, {r.digest for r in company_rows})

scored = sorted(
    ((sum(a * b for a, b in zip(query_vector, vectors[r.digest])), r) for r in company_rows),
    key=lambda pair: pair[0],
    reverse=True,
)
by_hand = pd.DataFrame(
    [{"cosine": round(s, 4), "fy": r.fiscal_year, "items": r.items, "chunk_id": r.chunk_id,
      "text": r.text[:90]} for s, r in scored[:5]]
)
by_hand

## 10 · The manifest — Stage 3's durable output

One row per chunk: metadata, the chunk text (without the contextual prefix — that
is for the encoder only), and the digest linking it to its vector. Together with the cache,
this is everything needed to rebuild the index. The index itself is disposable.

In [ ]:
manifest = pd.DataFrame(rows)
print(f"{len(manifest):,} rows, {manifest['digest'].nunique():,} distinct vectors")
# A chunk with two items counts under both.
manifest.explode("items").pivot_table(
    index="ticker", columns="items", values="chunk_id", aggfunc="count", fill_value=0
)

## 11 · One index document — manifest row + vector

This is exactly the JSON `build_index` sends to OpenSearch for one chunk
(vector shortened for display). `chunk_id` becomes the document `_id`, so rebuilding overwrites instead of duplicating.

In [ ]:
from filing_copilot.index import to_document

row = next(r for r in rows if r.digest == key)
document_json = to_document(row, vectors.get(key, cached))
{**document_json, "embedding": document_json["embedding"][:4] + ["..."], "text": row.text[:120] + "..."}

## 12 · Query the index — lexical, semantic, and filtered

- **BM25** matches words ("CECL", "Item 9A") — the half that doesn't paraphrase.
- **k-NN** matches meaning. faiss reports inner product as `1 + dot`, so subtract 1 to compare with cell 9.
- **k-NN + filter** scopes to one company. Its top 5 should match cell 9's hand-computed list.

Stage 4 fuses BM25 and k-NN by *rank* (RRF, ADR-0010), because their scores are on unrelated scales.

In [ ]:
import httpx
from filing_copilot.index import index_name

url = f"{settings.opensearch_url}/{index_name(settings.opensearch_index_prefix, 'item_aware')}/_search"
fields = ["ticker", "fiscal_year", "items", "chunk_id", "text"]

def search(query: dict, size: int = 5) -> pd.DataFrame:
    hits = httpx.post(url, json={"size": size, "_source": fields, "query": query}).json()["hits"]["hits"]
    return pd.DataFrame(
        [{"score": round(h["_score"], 4), "fy": h["_source"]["fiscal_year"], "ticker": h["_source"]["ticker"],
          "items": h["_source"]["items"], "chunk_id": h["_id"], "text": h["_source"]["text"][:90]} for h in hits]
    )

print("BM25 ----------------------------------------------------------")
display(search({"match": {"text": QUESTION}}))
print("k-NN, whole corpus ---------------------------------------------")
display(search({"knn": {"embedding": {"vector": query_vector, "k": 5}}}))
print(f"k-NN, ticker = {TICKER} ------------------------------------------")
filtered = search({"knn": {"embedding": {"vector": query_vector, "k": 5, "filter": {"term": {"ticker": TICKER}}}}})
display(filtered)

print("matches the by-hand ranking from cell 9:", list(filtered["chunk_id"]) == list(by_hand["chunk_id"]))

## What Stage 3 outputs

| Artifact | Durable? | Rebuilt by |
|---|---|---|
| Embedding cache — `(digest, vector)` Parquet parts | yes, append-only | `fc embed` (only missing digests) |
| Manifest — one row per chunk, with its digest | yes, replaced each run | `fc embed` |
| OpenSearch index | **no** — derived | `fc build-index`, ~14s, zero embeddings |

The index is the join of the other two on `digest`. Lose it and nothing is lost.
Stage 4 queries it both ways and fuses the rankings.

In [ ]:
encoder.close()